# Load data

In [1]:
import pandas as pd

SHORT = False

# Load data from parquet file
if SHORT:
    file_path_parquet = 'data/electricity_clean_short.parquet'
    metadata_path = 'data/metadata_clean_short.csv'
else:
    file_path_parquet = 'data/electricity_clean.parquet'
    metadata_path = 'data/metadata_clean.csv'

# Load the cleaned data
df_elec = pd.read_parquet(file_path_parquet)
df_meta = pd.read_csv(metadata_path)

# Quick check to ensure the index and columns are correct
print(df_elec.info())
# print(df_elec.head())

<class 'pandas.core.frame.DataFrame'>
DatetimeIndex: 17544 entries, 2016-01-01 00:00:00 to 2017-12-31 23:00:00
Columns: 1027 entries, Robin_public_Carolina to Mouse_health_Estela
dtypes: float64(1027)
memory usage: 137.6 MB
None


# Melt from wide to long

In [2]:
import pandas as pd
import time

# Start the timer
start_time = time.time()

print("Starting 'Melt' operation...")

df_start = df_elec
# df_start = df_elec.copy()
# 1. Transform to Long Format
df_long = df_start.reset_index().melt(
    id_vars='timestamp', 
    var_name='building_id', 
    value_name='meter_reading'
)

# 2. Memory Optimization
# This is the "magic" line that shrinks the file size
df_long['building_id'] = df_long['building_id'].astype('category')

# Stop the timer
end_time = time.time()

# --- Monitoring Report ---
duration = end_time - start_time
# .memory_usage(deep=True) gives the actual RAM footprint in bytes
mem_usage_mb = df_long.memory_usage(deep=True).sum() / (1024**2)

print("-" * 30)
print(f"Melt Completed in: {duration:.2f} seconds")
print(f"Final Row Count:   {len(df_long):,}")
print(f"Memory Usage:      {mem_usage_mb:.2f} MB")
print("-" * 30)

# Check the shape: Should be (Timestamps * Buildings, New Columns)
print(f"Dataframe Shape: {df_long.shape}")

# Inspect the first few rows
print("\n--- First 5 Rows (Head) ---")
display(df_long.head())

# Statistical Summary: 
# # Look at 'min' and 'max' for sin/cos columns—they MUST be between -1 and 1.
# print("\n--- Statistical Summary ---")
# display(df_long.describe())

# # Look at 10 random rows from across the entire 17M row set
# print("\n--- Random Data Sample ---")
# display(df_long.sample(10))

# # Verify the number of unique buildings still matches your cleaned set
# print(f"\nUnique Buildings: {df_long['building_id'].nunique()}")

Starting 'Melt' operation...
------------------------------
Melt Completed in: 2.64 seconds
Final Row Count:   18,017,688
Memory Usage:      309.39 MB
------------------------------
Dataframe Shape: (18017688, 3)

--- First 5 Rows (Head) ---


,timestamp,building_id,meter_reading
0,2016-01-01 00:00:00,Robin_public_Carolina,36.438
1,2016-01-01 01:00:00,Robin_public_Carolina,70.750
2,2016-01-01 02:00:00,Robin_public_Carolina,74.312
3,2016-01-01 03:00:00,Robin_public_Carolina,73.438
4,2016-01-01 04:00:00,Robin_public_Carolina,70.313


# Add weather data

In [3]:
# Using api weather data
import pandas as pd
import numpy as np
import time

# Create an isolated copy to work with in this cell
df_weather_combined = df_long
# df_weather_combined = df_long.copy()


# --- STEP 1: PRE-PROCESS WEATHER (Skinny Calculation) ---
print("Pre-processing weather features...")
df_weather = pd.read_csv('data/weather_data.csv')

# Calculate CDH/HDH and Rolling Temp on the unique weather hours
# Using the new column name: temperature_f
df_weather['CDH'] = (df_weather['temperature_f'] - 65).clip(lower=0)
df_weather['HDH'] = (65 - df_weather['temperature_f']).clip(lower=0)
df_weather['temp_roll_3h'] = df_weather.groupby('location_id')['temperature_f'].transform(
    lambda x: x.rolling(window=3).mean()
)

# Filter for lean columns and downcast to float32
# Using updated names: temperature_f and apparent_temperature_f
weather_cols = [
    'location_id', 'time', 'temperature_f', 
    'apparent_temperature_f', 'CDH', 'HDH', 'temp_roll_3h'
]
df_weather_skinny = df_weather[weather_cols].copy()

for col in df_weather_skinny.select_dtypes(include=['float64']).columns:
    df_weather_skinny[col] = pd.to_numeric(df_weather_skinny[col], downcast='float')

del df_weather # Cleanup raw weather data

# --- STEP 2: SPATIAL MAPPING ---
df_weather_meta = pd.read_csv('data/weather_metadata.csv')

def find_nearest_location(row, weather_df):
    distances = np.sqrt((weather_df['latitude'] - row['lat'])**2 + (weather_df['longitude'] - row['lng'])**2)
    return weather_df.loc[distances.idxmin(), 'location_id']

# Map building_id to location_id using metadata coordinates
df_meta['location_id'] = df_meta.apply(find_nearest_location, axis=1, weather_df=df_weather_meta)
location_map = df_meta[['building_id', 'location_id']]

# --- STEP 3: THE MERGE ---
start_time = time.time()
print("Merging weather features...")

# Join location IDs to electricity copy
df_weather_combined = df_weather_combined.merge(location_map, on='building_id', how='left')

# Convert electricity timestamp to Unix seconds for the join
df_weather_combined['time_unix'] = df_weather_combined['timestamp'].astype('int64') // 10**9

# Join the skinny weather features
df_weather_combined = df_weather_combined.merge(
    df_weather_skinny, 
    left_on=['location_id', 'time_unix'], 
    right_on=['location_id', 'time'], 
    how='left'
)

# Final Cleanup of join keys
df_weather_combined = df_weather_combined.drop(columns=['time', 'time_unix'])
del df_weather_skinny

print(f"Merge completed in {time.time() - start_time:.2f}s")
print(f"Final Memory Usage: {df_weather_combined.memory_usage(deep=True).sum() / 1024**2:.2f} MB")
print("-" * 30)
print(df_weather_combined.head(20))

Pre-processing weather features...
Merging weather features...
Merge completed in 7.23s
Final Memory Usage: 1935.24 MB
------------------------------
             timestamp            building_id  meter_reading  location_id  \
0  2016-01-01 00:00:00  Robin_public_Carolina         36.438           11   
1  2016-01-01 01:00:00  Robin_public_Carolina         70.750           11   
2  2016-01-01 02:00:00  Robin_public_Carolina         74.312           11   
3  2016-01-01 03:00:00  Robin_public_Carolina         73.438           11   
4  2016-01-01 04:00:00  Robin_public_Carolina         70.313           11   
5  2016-01-01 05:00:00  Robin_public_Carolina         70.250           11   
6  2016-01-01 06:00:00  Robin_public_Carolina         71.875           11   
7  2016-01-01 07:00:00  Robin_public_Carolina         68.250           11   
8  2016-01-01 08:00:00  Robin_public_Carolina         83.499           11   
9  2016-01-01 09:00:00  Robin_public_Carolina         98.938           11   
10 

# Time

In [4]:
import numpy as np
import time

start_time = time.time()
print("Engineering cyclical time features...")

df_time = df_weather_combined
# df_time = df_weather_combined.copy()

# 1. Extract base components
df_time['hour'] = df_time['timestamp'].dt.hour
df_time['day_week'] = df_time['timestamp'].dt.dayofweek
df_time['month'] = df_time['timestamp'].dt.month
# 5, 6 for saturday and sunday
# df_time['is_weekend'] = df_time['day_week'].isin([5, 6]).astype(int)
# 4, 5 for friday saturday
df_time['is_weekend'] = df_time['day_week'].isin([4, 5]).astype(int)

# # 2. Apply Cyclical Encoding (Sine/Cosine)
# # Hour (Period = 24)
# df_time['hour_sin'] = np.sin(2 * np.pi * df_time['hour'] / 24)
# df_time['hour_cos'] = np.cos(2 * np.pi * df_time['hour'] / 24)

# # Day of Week (Period = 7)
# df_time['day_sin'] = np.sin(2 * np.pi * df_time['day_week'] / 7)
# df_time['day_cos'] = np.cos(2 * np.pi * df_time['day_week'] / 7)

# # Month (Period = 12)
# # Subtract 1 so months 1-12 become 0-11 for smoother math
# df_time['month_sin'] = np.sin(2 * np.pi * (df_time['month'] - 1) / 12)
# df_time['month_cos'] = np.cos(2 * np.pi * (df_time['month'] - 1) / 12)

# # Downcast floats to save 50% space on the new columns
# float_cols = [c for c in df_time.columns if 'sin' in c or 'cos' in c]
# for col in float_cols:
#     df_time[col] = pd.to_numeric(df_time[col], downcast='float')

# Downcast integers (hour, day, month, is_weekend)
int_cols = ['hour', 'day_week', 'month', 'is_weekend']
for col in int_cols:
    df_time[col] = pd.to_numeric(df_time[col], downcast='integer')

duration = time.time() - start_time
mem_usage_mb = df_time.memory_usage(deep=True).sum() / (1024**2)

print("-" * 30)
print(f"Features Completed in: {duration:.2f} seconds")
print(f"Current Memory Usage:  {mem_usage_mb:.2f} MB")
print("-" * 30)

# Check the shape: Should be (Timestamps * Buildings, New Columns)
print(f"Dataframe Shape: {df_time.shape}")

# Inspect the first few rows
display(df_time.head())

# Statistical Summary: 
# Look at 'min' and 'max' for sin/cos columns—they MUST be between -1 and 1.
display(df_time.describe())

Engineering cyclical time features...
------------------------------
Features Completed in: 4.45 seconds
Current Memory Usage:  2003.97 MB
------------------------------
Dataframe Shape: (18017688, 13)


,timestamp,building_id,meter_reading,location_id,temperature_f,apparent_temperature_f,CDH,HDH,temp_roll_3h,hour,day_week,month,is_weekend
0,2016-01-01 00:00:00,Robin_public_Carolina,36.438,11,39.500000,33.200001,0.0,25.500000,40.866665,0,4,1,1
1,2016-01-01 01:00:00,Robin_public_Carolina,70.750,11,38.200001,32.799999,0.0,26.799999,39.466667,1,4,1,1
2,2016-01-01 02:00:00,Robin_public_Carolina,74.312,11,36.500000,31.500000,0.0,28.500000,38.066666,2,4,1,1
3,2016-01-01 03:00:00,Robin_public_Carolina,73.438,11,34.799999,29.700001,0.0,30.200001,36.500000,3,4,1,1
4,2016-01-01 04:00:00,Robin_public_Carolina,70.313,11,33.099998,27.700001,0.0,31.900000,34.799999,4,4,1,1


,timestamp,meter_reading,location_id,temperature_f,apparent_temperature_f,CDH,HDH,temp_roll_3h,hour,day_week,month,is_weekend
count,18017688,1.766446e+07,1.801769e+07,1.801769e+07,1.801769e+07,1.801769e+07,1.801769e+07,1.801769e+07,1.801769e+07,1.801769e+07,1.801769e+07,1.801769e+07
mean,2016-12-31 11:29:59.999995136,1.514586e+02,5.301850e+00,5.776499e+01,5.468807e+01,4.358640e+00,1.159365e+01,5.776575e+01,1.150000e+01,3.008208e+00,6.519836e+00,2.872777e-01
min,2016-01-01 00:00:00,1.000000e-04,1.000000e+00,-2.600000e+01,-3.580000e+01,0.000000e+00,0.000000e+00,-2.563333e+01,0.000000e+00,0.000000e+00,1.000000e+00,0.000000e+00
25%,2016-07-01 17:45:00,2.029000e+01,3.000000e+00,4.530000e+01,3.910000e+01,0.000000e+00,0.000000e+00,4.540000e+01,5.750000e+00,1.000000e+00,4.000000e+00,0.000000e+00
50%,2016-12-31 11:30:00,6.250000e+01,4.000000e+00,5.770000e+01,5.440000e+01,0.000000e+00,7.300000e+00,5.776667e+01,1.150000e+01,3.000000e+00,7.000000e+00,0.000000e+00
75%,2017-07-02 05:15:00,1.663100e+02,7.000000e+00,7.070000e+01,7.090000e+01,5.700000e+00,1.970000e+01,7.066666e+01,1.725000e+01,5.000000e+00,1.000000e+01,1.000000e+00
max,2017-12-31 23:00:00,5.701000e+03,1.200000e+01,1.179000e+02,1.187000e+02,5.290000e+01,9.100000e+01,1.175000e+02,2.300000e+01,6.000000e+00,1.200000e+01,1.000000e+00
std,NaN,2.662473e+02,3.227865e+00,1.839355e+01,2.195968e+01,7.418224e+00,1.339222e+01,1.832858e+01,6.922187e+00,2.000667e+00,3.449551e+00,4.524922e-01


# Final touchups

# Metadata stuff

In [5]:
import pandas as pd

df_final = df_time
# df_final = df_time.copy()

# 2. Map metadata features (sqft, sub_type, and year_built)
# Setting the index on df_meta once for efficient mapping
meta_map = df_meta.set_index('building_id')

# 1. Mapping from Metadata
df_final['sqft'] = df_final['building_id'].map(meta_map['sqft'])
df_final['primary_space_usage'] = df_final['building_id'].map(meta_map['primaryspaceusage'])
df_final['sub_type'] = df_final['building_id'].map(meta_map['sub_primaryspaceusage'])
df_final['year_built'] = df_final['building_id'].map(meta_map['yearbuilt'])
df_final['number_of_floors'] = df_final['building_id'].map(meta_map['numberoffloors']) # Added this

# 2. Downcast and Optimize
# Categories for strings
df_final['primary_space_usage'] = df_final['primary_space_usage'].astype('category')
df_final['sub_type'] = df_final['sub_type'].astype('category')

# Float32 for numbers (preserves NaNs in year_built and number_of_floors)
df_final['sqft'] = pd.to_numeric(df_final['sqft'], downcast='float')
df_final['year_built'] = pd.to_numeric(df_final['year_built'], downcast='float')
df_final['number_of_floors'] = pd.to_numeric(df_final['number_of_floors'], downcast='float')

# Check for missing values in the new features
print("-" * 30)
print(f"Rows missing sqft:       {df_final['sqft'].isna().sum()}")
print(f"Rows missing primary_space_usage:   {df_final['primary_space_usage'].isna().sum()}")
print(f"Rows missing sub_type:   {df_final['sub_type'].isna().sum()}")
print(f"Rows missing year_built: {df_final['year_built'].isna().sum()} ({df_final['year_built'].isna().mean()*100:.1f}%)")
print("-" * 30)
print("Feature Engineering Complete: Added primary_space_usage, sub_type, sqft, and year_built.")

display(df_final[['building_id', 'primary_space_usage', 'sub_type', 'sqft', 'year_built']].head())

------------------------------
Rows missing sqft:       0
Rows missing primary_space_usage:   0
Rows missing sub_type:   0
Rows missing year_built: 8526384 (47.3%)
------------------------------
Feature Engineering Complete: Added primary_space_usage, sub_type, sqft, and year_built.


,building_id,primary_space_usage,sub_type,sqft,year_built
0,Robin_public_Carolina,Public services,Library,118231.0,NaN
1,Robin_public_Carolina,Public services,Library,118231.0,NaN
2,Robin_public_Carolina,Public services,Library,118231.0,NaN
3,Robin_public_Carolina,Public services,Library,118231.0,NaN
4,Robin_public_Carolina,Public services,Library,118231.0,NaN


# Drop NAN meter readings

In [6]:
# --- FINAL CLEANUP BEFORE EXPORT ---

# This ensures the training notebook doesn't crash on missing readings
df_final = df_final.dropna(subset=['meter_reading']).copy()

# 2. Final Type Cast (Ensuring nothing reverted to float64)
# This keeps your 17M rows from bloating the Parquet file.
for col in df_final.select_dtypes(include=['float64']).columns:
    df_final[col] = pd.to_numeric(df_final[col], downcast='float')

print(f"Final Row Count for Export: {len(df_final):,}")
print(f"Missing meter readings removed. Ready for training.")

# This will show you exactly which columns have NaNs and how many
print("--- NaN Count Per Column ---")
print(df_final.isna().sum())

Final Row Count for Export: 17,664,462
Missing meter readings removed. Ready for training.
--- NaN Count Per Column ---
timestamp                        0
building_id                      0
meter_reading                    0
location_id                      0
temperature_f                    0
apparent_temperature_f           0
CDH                              0
HDH                              0
temp_roll_3h                     0
hour                             0
day_week                         0
month                            0
is_weekend                       0
sqft                             0
primary_space_usage              0
sub_type                         0
year_built                 8415148
number_of_floors          12987030
dtype: int64


# Save to file

In [7]:
import os

# --- PATH LOGIC ---
# Uses the 'SHORT' variable from your first cell to name the file
if SHORT:
    OUTPUT_NAME = 'engineered_short.parquet'
else:
    OUTPUT_NAME = 'engineered.parquet'

DATA_DIR = 'data'
save_path = os.path.join(DATA_DIR, OUTPUT_NAME)

if not os.path.exists(DATA_DIR):
    os.makedirs(DATA_DIR)

print(f"Exporting {'SHORT' if SHORT else 'FULL'} dataset to Parquet...")
start_save = time.time()

# --- THE SAVE ---
# We save WITHOUT the index because 'timestamp' is now a column, 
# and 'building_id' is a feature.
df_final.to_parquet(save_path, engine='pyarrow', index=False)

print("-" * 30)
print(f"File Saved: {save_path}")
print(f"Save Time:  {time.time() - start_save:.2f} seconds")
print(f"Final Row Count: {len(df_final):,}")
print("-" * 30)

# Verify the file exists and check size
file_size = os.path.getsize(save_path) / (1024**2)
print(f"File Size on Disk: {file_size:.2f} MB")

Exporting FULL dataset to Parquet...
------------------------------
File Saved: data\engineered.parquet
Save Time:  11.79 seconds
Final Row Count: 17,664,462
------------------------------
File Size on Disk: 155.36 MB
